In [3]:
# Parameters
Month = 3
Year = 2569
session = "g"


In [4]:
import pandas as pd
import os
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload
from oauth2client.service_account import ServiceAccountCredentials
from pathlib import Path

current_path = Path.cwd()

# Month = 2
# session = 'g' # /// g คือชุดทั่วไป /// u คือชุดนอกเขต
# Year = 2569

Web_Data_SsoTpso = 1 # /// อยากให้รันข้อมูลสินค้นให้กดหนึ่ง /// ถ้าไม่รันกดศูนย์
Web_Shop_12C = 0
text = session

In [5]:
masterF =Path(*current_path.parts[:-1]) / "01-database" / "Master_CPI" / "Real_Master_CPI.xlsx"
# Ensure the file exists before reading
try:
    Real_master = pd.read_excel(masterF)
    RM = Real_master[['รหัส', 'รายการ', 'ผู้ดูแล', 'กำหนดให้เก็บ', 'รายละเอียด','G/L','U']].rename(columns={'รหัส': 'CODE7'})
    RM["CODE7"] = RM["CODE7"].astype(str).str.zfill(7)
    RM = RM.dropna(subset=['กำหนดให้เก็บ'])
    RM['กำหนดให้เก็บ'] = pd.to_numeric(RM['กำหนดให้เก็บ'], errors='coerce')
    RM = RM[RM['กำหนดให้เก็บ'] > 0]    
    RM['กำหนดให้เก็บ'] = RM['กำหนดให้เก็บ'].astype(int)

    if text =='g':
        RM = RM[RM['G/L']>0]
    elif text == 'u':
        RM = RM[RM['U']>0]
    RM = RM[['CODE7', 'รายการ', 'ผู้ดูแล', 'กำหนดให้เก็บ', 'รายละเอียด']]
    rm_gits = RM['CODE7'].tolist()  

    RM = RM.reset_index(drop=True)
    # print(RM.head())

except FileNotFoundError:
    print(f"File {masterF} not found.")

rm_gits = RM['CODE7'].tolist()  

# RM
len(set(rm_gits))

440

In [6]:
if session == 'g':
    mg_ = pd.read_excel(current_path.parent/ "01-database" / "Data_G_Month" / f"cpi{session}_month_{Year}-{Month}.xlsx", engine='calamine')
    wg_ = pd.read_excel(current_path.parent/ "01-database" / "Data_G_Week" / f"cpi{session}_week_{Year}-{Month}.xlsx", engine='calamine')

elif session == 'u':  
    mu_ = pd.read_excel(current_path.parent/ "01-database" / "Data_U_Month" / f"cpi{session}_month_{Year}-{Month}.xlsx", engine='calamine')
    wu_ = pd.read_excel(current_path.parent/ "01-database" / "Data_U_Week" / f"cpi{session}_week_{Year}-{Month}.xlsx", engine='calamine')
    # wu_ = pd.read_excel(Main_path /"Data_U_Week"/ f"cpi{session}_week_{Year}-{Month}.xlsx", engine='calamine')


In [7]:
# mg_

In [8]:
current_path.parent/ "01-database" / "Data_U_Month" / f"cpi{session}_month_{Year}-{Month}.xlsx"

WindowsPath('d:/Coding/WorkFlow/01-database/Data_U_Month/cpig_month_2569-3.xlsx')

In [9]:
if session == 'g':
    mg = mg_.copy()
    wg = wg_.copy()
    mg.insert(0, 'ชุด', 'อ.เมือง')    
    wg.insert(0, 'ชุด', 'อ.เมือง')
elif session == 'u':  
    mu = mu_.copy()
    wu = wu_.copy()
    mu.insert(0, 'ชุด', 'นอกเขต')
    wu.insert(0, 'ชุด', 'นอกเขต')


# --- ต่อจาก Code ของคุณ ---

# 1. เลือกว่าจะใช้ชุดข้อมูลไหนตรวจสอบ (ตามตัวแปร text)
if text == 'g':
    # รวมข้อมูล General (รายเดือน + รายสัปดาห์)
    df_actual = pd.concat([mg, wg], ignore_index=True)

# คำนวณ Flag

    target_set_name = 'ชุดทั่วไป'
elif text == 'u':
    # รวมข้อมูล Rural/Urban (รายเดือน + รายสัปดาห์)
    df_actual = pd.concat([mu, wu], ignore_index=True)
    target_set_name = 'ชุดนอกเขต'
else:
    raise ValueError("ตัวแปร text ต้องเป็น 'g' หรือ 'u' เท่านั้น")

df = df_actual.copy()

# จัด Format รหัส
df["COMMODITY_CODE"] = df["COMMODITY_CODE"].astype(str).str.zfill(16)
df["DILLER_CODE"]    = df["DILLER_CODE"].astype(str).str.zfill(10)
df["CODE7"]  = df["COMMODITY_CODE"].astype(str).str[:7]
df["CODE7"] = df["COMMODITY_CODE"].str[0:7]
df["CODE9"] = df["COMMODITY_CODE"].str[7:16]
df.sort_values(by=['COMMODITY_CODE'], inplace=True)
df.reset_index(drop=True)

import pandas as pd
import glob
import os
current_dir = Path.cwd()
# ค้นหาไฟล์ทั้งหมดที่ขึ้นต้นด้วย shop_data และลงท้ายด้วย .xlsx
files_shop = glob.glob(str(current_dir.parent / "01-database" / "Shop_BKK" / "shop_data*.xlsx"))

ds_ = pd.read_excel(files_shop [0], skiprows=2, engine='calamine')
# Dealer Group
ds = ds_[["รหัสร้าน", "กลุ่ม"]].rename(columns={"รหัสร้าน": "DILLER_CODE"})
ds["DILLER_CODE"] = ds["DILLER_CODE"].astype(str).str.zfill(10)
ds["กลุ่ม"] = ds["กลุ่ม"].replace(["-", " "], "", regex=True)
df = df.merge(ds, on=["DILLER_CODE"], how="left")

df.loc[df['PROVINCE_NAME'] != 'กรุงเทพมหานคร', 'GROUP'] = df['PROVINCE_NAME']
df.loc[df['PROVINCE_NAME'] == 'กรุงเทพมหานคร', 'GROUP'] = 'กทม.'+df['กลุ่ม']
df = df.reindex(columns=[ 'CODE7', 'GROUP'])
df.to_clipboard()


In [10]:
Detail = RM[['CODE7','รายการ','กำหนดให้เก็บ','รายละเอียด']].copy()
Detail

,CODE7,รายการ,กำหนดให้เก็บ,รายละเอียด
0,1111001,ข้าวสารเจ้า,12,2แหล่ง (1หอมมะลิแบบตัก/2หอมมะลิแบบถุง/1ขาวเสาไ...
1,1111002,ข้าวสารเหนียว,4,2แหล่ง (1ข้าวเหนียวถัง/1ข้าวเหนียวถุง)
2,1112104,แป้งทอดกรอบ,4,2แหล่ง 2สเปค
3,1112202,วุ้นเส้น,8,2แหล่ง (2ขนาดเล็ก/2ขนาดใหญ่)
4,1112203,เต้าหู้,6,2แหล่ง (2เต้าหู้หลอด/1เต้าหู้แผ่นแข็ง)
...,...,...,...,...
435,6311002,ค่าอาหารสำหรับตักบาตร,2,2แหล่ง 1สเปค
436,7100001,บุหรี่,8,2แหล่ง (2บุหรี่นอก/2บุหรี่ใน)
437,7210001,เบียร์,8,2แหล่ง (2เบียร์นอก/2เบียร์ใน)
438,7220001,ไวน์,8,2แหล่ง (2ไวน์นอก/2ไวน์ใน)


In [11]:
current_path = Path.cwd()
SOURCE_FILE_PATH = Path(*current_path.parts[:-1]) / "01-database" / "Master_CPI"

province_list_exist = pd.read_excel(SOURCE_FILE_PATH/'Real_Master_CPI.xlsx', engine='calamine', sheet_name='จังหวัดใช้งาน')

if text == 'g':
    province_list_exist = province_list_exist[province_list_exist['ชุดCPI']=="ชุดอำเภอเมือง"]
    province_list = province_list_exist['จังหวัด/กลุ่ม'].dropna().tolist()
elif text == 'u':
    province_list_exist = province_list_exist[province_list_exist['ชุดCPI']=="ชุดนอกเขตเมือง"]
    province_list = province_list_exist['จังหวัด/กลุ่ม'].dropna().tolist()
text    


'g'

In [12]:
print(len(df['CODE7'].unique()))
print(len(rm_gits))
# หาตัวที่มีใน df['CODE7'] แต่ไม่มีใน rm_gits
diff = set(df['CODE7'].unique()) - set(rm_gits)
print(f"รหัสที่เกินมาคือ: {diff}")
# หาตัวที่มีใน df['CODE7'] แต่ไม่มีใน rm_gits
diff = set(set(rm_gits) - set(df['CODE7'].unique()))

print(f"รหัสที่เกินมาคือ: {diff}")

unnion_codes = set(df['CODE7'].unique()).union(set(rm_gits))
print(f"รหัสทั้งหมดที่ควรมี: {len(unnion_codes)}")

511
440
รหัสที่เกินมาคือ: {'5211005', '4112001', '1160004', '3600006', '1210007', '1142118', '1121212', '5220003', '4300002', '1142123', '5120007', '4111007', '2213001', '4220002', '6221203', '1220003', '2123003', '1122203', '1141124', '1141130', '1121208', '2212001', '1210009', '1132005', '5410007', '2211002', '6150004', '5110013', '2132002', '2221002', '5220011', '5120012', '5110003', '5230003', '4220001', '1123502', '4111020', '2125004', '5220007', '4210006', '5220004', '3150001', '5230008', '5310002', '2113006', '5220006', '5330004', '1141205', '5220008', '5110011', '1220005', '3120001', '4220004', '5220013', '1141125', '1112206', '5211002', '4121501', '1142122', '2123010', '4220003', '2223001', '5220010', '6124006', '1210013', '3130003', '6150005', '4300003', '2125002', '5110010', '3600007'}
รหัสที่เกินมาคือ: set()
รหัสทั้งหมดที่ควรมี: 511


In [13]:
# ปรับปรุง: ถ้ามีคำว่า 'กลุ่ม' ให้เติม 'กทม.' ไว้ข้างหน้า ถ้าไม่มีให้ใช้ชื่อเดิม
province_list = [f"กทม.{name}" if "กลุ่ม" in name else name for name in province_list]
province_list

['กทม.กลุ่มจตุจักร',
 'กทม.กลุ่มบางกะปิ',
 'กทม.กลุ่มบางบอน',
 'กทม.กลุ่มบางประกอก',
 'กทม.กลุ่มบางแค',
 'กทม.กลุ่มพระโขนง',
 'กทม.กลุ่มพรานนก',
 'กทม.กลุ่มมีนบุรี',
 'กทม.กลุ่มรุ่งเจริญพระราม3(บางรักเดิม)',
 'กทม.กลุ่มวงเวียนใหญ่',
 'กทม.กลุ่มสะพานใหม่',
 'กทม.กลุ่มห้วยขวาง',
 'กทม.กลุ่มเตาปูน',
 'กทม.กลุ่มเทเวศร์',
 'กระบี่',
 'กาญจนบุรี',
 'กาฬสินธุ์',
 'ขอนแก่น',
 'จันทบุรี',
 'ฉะเชิงเทรา',
 'ชลบุรี',
 'ตรัง',
 'ตาก',
 'นครนายก',
 'นครปฐม',
 'นครพนม',
 'นครราชสีมา',
 'นครศรีธรรมราช',
 'นครสวรรค์',
 'นนทบุรี',
 'นราธิวาส',
 'น่าน',
 'บึงกาฬ',
 'ปทุมธานี',
 'ประจวบคีรีขันธ์',
 'ปราจีนบุรี',
 'พระนครศรีอยุธยา',
 'พะเยา',
 'พังงา',
 'พิษณุโลก',
 'ภูเก็ต',
 'มุกดาหาร',
 'ยะลา',
 'ระนอง',
 'ระยอง',
 'ราชบุรี',
 'ร้อยเอ็ด',
 'ลพบุรี',
 'ลำปาง',
 'ศรีสะเกษ',
 'สกลนคร',
 'สงขลา',
 'สมุทรปราการ',
 'สระแก้ว',
 'สิงห์บุรี',
 'สุพรรณบุรี',
 'สุราษฎร์ธานี',
 'สุรินทร์',
 'หนองคาย',
 'อุดรธานี',
 'อุตรดิตถ์',
 'อุบลราชธานี',
 'เชียงราย',
 'เชียงใหม่',
 'เพชรบุรี',
 'เพชรบูรณ์',
 'เลย',
 'แพร่',
 

In [14]:
import pandas as pd
import numpy as np

# ... (Previous code remains the same up to unitGROUP filtering) ...

unit7 = df['CODE7'].unique()
unitGROUP = df['GROUP'].unique()    
unitGROUP = unitGROUP[np.isin(unitGROUP, province_list)]

# --- FIX STARTS HERE ---
import pandas as pd
import numpy as np

# ... (โค้ดส่วนเตรียม unitGROUP และ all_combinations เหมือนเดิม) ...

# --- แก้ไขส่วนนี้ (FIX STARTS HERE) ---

# 1. สร้างตารางแม่แบบทุกคู่ (เหมือนเดิม)
all_combinations = pd.MultiIndex.from_product(
    [rm_gits, unitGROUP], 
    names=['CODE7', 'GROUP']
).to_frame(index=False)

# 2. คำนวณ "ค่าจริง" จาก df (เปลี่ยนจากใส่เลข 1 เป็นการนับจำนวน)
# ใช้ groupby เพื่อดูว่า รหัสนี้ในจังหวัดนี้ มีกี่แถว (Count)
# หรือถ้าต้องการ "ราคาเฉลี่ย" ให้เปลี่ยน .size() เป็น ['AVG'].mean()
actual_data = df.groupby(['CODE7', 'GROUP']).size().reset_index(name='เก็บมาได้')

# 3. Merge ข้อมูลจริงเข้ากับตารางแม่แบบ
# Left join เพื่อคงโครงสร้าง Master x Province ไว้
merged_data = all_combinations.merge(actual_data, on=['CODE7', 'GROUP'], how='left')

# เติม 0 ให้ช่องที่ไม่มีข้อมูล (คือหาไม่เจอเลย = 0)
merged_data['เก็บมาได้'] = merged_data['เก็บมาได้'].fillna(0).astype(int)

# --- ส่วนสร้าง Save 1, 2, 3 (Logic เดิม แต่ค่าข้างในจะเป็นจำนวนจริงแล้ว) ---

# Sheet 1: Matrix
crosstab_result = merged_data.pivot(index='CODE7', columns='GROUP', values='เก็บมาได้')
save1 = crosstab_result.reset_index()
save1.columns.name = None 
save1 = save1.merge(Detail, on=['CODE7'], how='left')

cols_to_move = ['รายการ', 'กำหนดให้เก็บ', 'รายละเอียด']
remaining_cols = [c for c in save1.columns if c != 'CODE7' and c not in cols_to_move]
new_column_order = ['CODE7'] + cols_to_move + remaining_cols

save1 = save1[new_column_order]
save1.insert(0, 'ปี/เดือน', f'{Year}/{Month}')

# Sheet 2: Long Format (แสดงค่าจริง)
save2 = merged_data.copy()
save2 = save2.merge(Detail, on=['CODE7'], how='left')

selected_columns = ['CODE7', 'รายการ', 'GROUP', 'เก็บมาได้', 'กำหนดให้เก็บ', 'รายละเอียด']  
available_cols = [c for c in selected_columns if c in save2.columns]
save2 = save2[available_cols]
save2.insert(0, 'ปี/เดือน', f'{Year}/{Month}')

# Sheet 3: Missing Only (เฉพาะที่ได้ 0)
save3 = save2[save2['เก็บมาได้'] == 0].copy()

# แสดงผลทดสอบ
print("ตัวอย่างข้อมูลใน Save2 (แสดงจำนวนที่เก็บได้จริง):")
display(save2.head())
save3

ตัวอย่างข้อมูลใน Save2 (แสดงจำนวนที่เก็บได้จริง):


,ปี/เดือน,CODE7,รายการ,GROUP,เก็บมาได้,กำหนดให้เก็บ,รายละเอียด
0,2569/3,1111001,ข้าวสารเจ้า,กทม.กลุ่มห้วยขวาง,12,12,2แหล่ง (1หอมมะลิแบบตัก/2หอมมะลิแบบถุง/1ขาวเสาไ...
1,2569/3,1111001,ข้าวสารเจ้า,กทม.กลุ่มพระโขนง,13,12,2แหล่ง (1หอมมะลิแบบตัก/2หอมมะลิแบบถุง/1ขาวเสาไ...
2,2569/3,1111001,ข้าวสารเจ้า,กทม.กลุ่มรุ่งเจริญพระราม3(บางรักเดิม),12,12,2แหล่ง (1หอมมะลิแบบตัก/2หอมมะลิแบบถุง/1ขาวเสาไ...
3,2569/3,1111001,ข้าวสารเจ้า,กทม.กลุ่มเทเวศร์,13,12,2แหล่ง (1หอมมะลิแบบตัก/2หอมมะลิแบบถุง/1ขาวเสาไ...
4,2569/3,1111001,ข้าวสารเจ้า,เชียงใหม่,12,12,2แหล่ง (1หอมมะลิแบบตัก/2หอมมะลิแบบถุง/1ขาวเสาไ...


,ปี/เดือน,CODE7,รายการ,GROUP,เก็บมาได้,กำหนดให้เก็บ,รายละเอียด
103,2569/3,1111002,ข้าวสารเหนียว,พังงา,0,4,2แหล่ง (1ข้าวเหนียวถัง/1ข้าวเหนียวถุง)
874,2569/3,1121106,เครื่องในวัว,กทม.กลุ่มสะพานใหม่,0,2,2แหล่ง 1สเปค
883,2569/3,1121106,เครื่องในวัว,กทม.กลุ่มบางกะปิ,0,2,2แหล่ง 1สเปค
1391,2569/3,1121211,แฮม/เบคอน,แม่ฮ่องสอน,0,4,2แหล่ง 2สเปค
1552,2569/3,1123101,ปลาช่อน,พังงา,0,4,2แหล่ง 2สเปค
...,...,...,...,...,...,...,...
30344,2569/3,7230001,สุรา,กทม.กลุ่มเตาปูน,0,8,2แหล่ง (2สุรานอก/2สุราใน)
30351,2569/3,7230001,สุรา,ภูเก็ต,0,8,2แหล่ง (2สุรานอก/2สุราใน)
30354,2569/3,7230001,สุรา,สุพรรณบุรี,0,8,2แหล่ง (2สุรานอก/2สุราใน)
30356,2569/3,7230001,สุรา,ยะลา,0,8,2แหล่ง (2สุรานอก/2สุราใน)


In [15]:
import xlsxwriter

# ==========================================
# 1. ตั้งชื่อไฟล์ตาม Text
# ==========================================
if text == 'g':
    file_name_export = "การเก็บวิเคราะห์การเก็บ โดยละเอียด ชุดทั่วไป.xlsx"
    print(">>> Mode: General Group (ชุดทั่วไป)")
elif text == 'u':
    file_name_export = "การเก็บวิเคราะห์การเก็บ โดยละเอียด ชุดนอกเขต.xlsx"
    print(">>> Mode: Urban/Rural Group (ชุดนอกเขต)")
else:
    file_name_export = "Analysis_Unknown.xlsx"

# ==========================================
# 3. Export & Format (ทำให้สวยงาม)
# ==========================================
print(f"กำลังบันทึกไฟล์: {file_name_export} ...")
import pandas as pd
import numpy as np
import xlsxwriter
from xlsxwriter.utility import xl_col_to_name

# ... (ส่วน Logic การเตรียมข้อมูล 2.1 - 2.4 คงเดิมตาม Code ก่อนหน้า) ...

# ==========================================
# 3. Export & Format (ฉบับจัดกลาง + บีบคอลัมน์ + เทียบเป้าหมาย)
# ==========================================
print(f"กำลังบันทึกไฟล์: {file_name_export} ...")

with pd.ExcelWriter(file_name_export, engine='xlsxwriter') as writer:
    
    def format_sheet(df, sheet_name, is_matrix=False):
        df.to_excel(writer, sheet_name=sheet_name, index=False)
        workbook  = writer.book
        worksheet = writer.sheets[sheet_name]
        
        # --- Styles Definition ---
        # 1. หัวตาราง (Header): สีน้ำเงิน, ตัวขาว, ตรงกลาง, ตัดคำ
        header_fmt = workbook.add_format({
            'bold': True, 'text_wrap': True, 'align': 'center', 'valign': 'vcenter',
            'fg_color': '#2F75B5', 'font_color': 'white', 'border': 1
        })
        
        # 2. ตัวเลข/รหัส (Numeric): อยู่ตรงกลาง
        center_fmt = workbook.add_format({'align': 'center', 'valign': 'vcenter', 'border': 1})
        
        # 3. ข้อความยาว (Text): ตัดคำ, ชิดซ้าย
        text_wrap_fmt = workbook.add_format({'text_wrap': True, 'valign': 'top', 'border': 1})
        
        # 4. สีแจ้งเตือน (Conditional Formatting)
        red_fmt = workbook.add_format({'bg_color': '#FFC7CE', 'font_color': '#9C0006'})   # ขาด (0)
        green_fmt = workbook.add_format({'bg_color': '#C6EFCE', 'font_color': '#006100'}) # ครบตามเป้า (>= กำหนดให้เก็บ)
        yellow_fmt = workbook.add_format({'bg_color': '#FFEB9C', 'font_color': '#9C5700'}) # มีของแต่ไม่ครบ (< กำหนดให้เก็บ)

        # --- Write Header ---
        for col_num, value in enumerate(df.columns.values):
            worksheet.write(0, col_num, value, header_fmt)
            
        # --- Set Column Width & Format ---
# --- Set Column Width & Format ---
        if is_matrix:
            # === [Sheet 1: Matrix Check] (ขยับ Index รองรับคอลัมน์ ปี/เดือน) ===
            
            # Col 0: ปี/เดือน (กว้าง 12, กลาง) -> ใหม่!
            worksheet.set_column(0, 0, 12, center_fmt)
            
            # Col 1: CODE7 (เดิม 0 -> 1)
            worksheet.set_column(1, 1, 10, center_fmt)
            
            # Col 2: รายการ (เดิม 1 -> 2)
            worksheet.set_column(2, 2, 25, text_wrap_fmt)
            
            # Col 3: กำหนดให้เก็บ (เดิม 2 -> 3) -> ตอนนี้อยู่ที่คอลัมน์ D
            worksheet.set_column(3, 3, 8, center_fmt)
            
            # Col 4: รายละเอียด (เดิม 3 -> 4)
            worksheet.set_column(4, 4, 30, text_wrap_fmt)
            
            # ** Col 5 ถึงจบ: จังหวัด (เดิมเริ่ม 4 -> 5) **
            last_col = len(df.columns) - 1
            if last_col >= 5:
                worksheet.set_column(5, last_col, 6, center_fmt)
            
            # Freeze Panes (ตรึงหลังรายละเอียด คือหลังคอลัมน์ index 4)
            worksheet.freeze_panes(1, 5)
            
            # --- Conditional Formatting (เทียบกับเป้าหมาย) ---
            # ข้อมูลจังหวัดเริ่มที่ Col 5 (F)
            # เป้าหมาย (กำหนดให้เก็บ) อยู่ที่ Col 3 ($D)
            
            target_col_letter = '$D' # คอลัมน์ "กำหนดให้เก็บ"
            start_data_col_idx = 5   # คอลัมน์เริ่มข้อมูลจังหวัด (F)
            start_data_col_letter = xl_col_to_name(start_data_col_idx) # ได้ 'F'
            
            # 1. สีแดง: ถ้าเท่ากับ 0
            worksheet.conditional_format(1, start_data_col_idx, len(df), last_col, {
                'type': 'cell', 'criteria': '=', 'value': 0, 'format': red_fmt
            })
            
            # 2. สีเขียว: ถ้า >= คอลัมน์ "กำหนดให้เก็บ" (Column D)
            # สูตร: =F2>=$D2
            formula_green = f'={start_data_col_letter}2>={target_col_letter}2' 
            
            worksheet.conditional_format(1, start_data_col_idx, len(df), last_col, {
                'type': 'formula', 'criteria': formula_green, 'format': green_fmt
            })

            # 3. สีเหลือง: ถ้า > 0 แต่น้อยกว่าเป้าหมาย
            # สูตร: =AND(F2>0, F2<$D2)
            formula_yellow = f'=AND({start_data_col_letter}2>0, {start_data_col_letter}2<{target_col_letter}2)'
            worksheet.conditional_format(1, start_data_col_idx, len(df), last_col, {
                'type': 'formula', 'criteria': formula_yellow, 'format': yellow_fmt
            })

        else:
            # === [Sheet 2 & 3: Data List] ===
            for i, col_name in enumerate(df.columns):
                width = 15
                fmt = text_wrap_fmt
                
                if col_name in ['CODE7', 'เก็บมาได้', 'กำหนดให้เก็บ']:
                    width = 10; fmt = center_fmt
                elif col_name == 'รายการ':
                    width = 30
                elif col_name == 'รายละเอียด':
                    width = 40
                elif col_name == 'GROUP':
                    width = 20
                
                worksheet.set_column(i, i, width, fmt)
            
            # Freeze Panes
            freeze_idx = 3 # ตรึงหลัง GROUP
            worksheet.freeze_panes(1, freeze_idx)

    # --- Execute ---
    format_sheet(save1, 'ตารางสรุปรายจังหวัด', is_matrix=True)
    format_sheet(save2, 'รายการทั้งหมด', is_matrix=False)
    format_sheet(save3, 'รายการที่ยังขาด', is_matrix=False)

print(f"✅ เสร็จสิ้น! บันทึกไฟล์เรียบร้อยที่: {file_name_export}")


>>> Mode: General Group (ชุดทั่วไป)
กำลังบันทึกไฟล์: การเก็บวิเคราะห์การเก็บ โดยละเอียด ชุดทั่วไป.xlsx ...
กำลังบันทึกไฟล์: การเก็บวิเคราะห์การเก็บ โดยละเอียด ชุดทั่วไป.xlsx ...
✅ เสร็จสิ้น! บันทึกไฟล์เรียบร้อยที่: การเก็บวิเคราะห์การเก็บ โดยละเอียด ชุดทั่วไป.xlsx


In [16]:
# import json

# # 1. แปลง save1 เป็น JSON 
# # orient='records' จะสร้างรูปแบบ [ {"col1": val1, "col2": val2}, ... ] ที่ HTML ของคุณต้องการ
# # force_ascii=False เพื่อให้ภาษาไทยไม่กลายเป็นรหัสตัวเลข
# json_result = save1.to_json(orient='records', force_ascii=False)

# # 2. บันทึกลงไฟล์ชื่อ my_data2.json
# with open('my_data2.json', 'w', encoding='utf-8') as f:
#     f.write(json_result)

# print("✅ แปลง save1 เป็น my_data2.json สำเร็จแล้ว!")

In [17]:
# import json

# # แปลง save1 เป็นรูปแบบ JSON (Records) เพื่อให้ตรงกับโครงสร้างที่ HTML ต้องการ
# # orient='records' จะสร้าง List ของ Dictionary [{...}, {...}]
# # force_ascii=False เพื่อให้แสดงภาษาไทยได้อย่างถูกต้อง
# json_data = save1.to_json(orient='records', force_ascii=False)

# # บันทึกไฟล์ชื่อ my_data2.json
# with open('my_data3.json', 'w', encoding='utf-8') as f:
#     f.write(json_data)

# print("✅ บันทึกไฟล์ JSON สำหรับแสดงผลบนเว็บเรียบร้อยแล้ว!")
# xxxx

In [18]:
# import json

# # เตรียมข้อมูลสำหรับ JSON
# export_package = {
#     "metadata": {
#         "year_month": f"{Year}/{Month}",
#         "display_date": f"ข้อมูลประจำเดือน {Month}/{Year}" 
#     },
#     "data": save1.to_dict(orient='records')
# }

# # บันทึกลงไฟล์ my_data2.json (แก้ไขขีดกลางเป็น Underscore)
# with open('my_data5.json', 'w', encoding='utf-8') as f:
#     json.dump(export_package, f, ensure_ascii=False, indent=2)

# print(f"✅ สร้าง JSON สำเร็จสำหรับรอบ {Year}/{Month}")

In [19]:
import json
import os
from pathlib import Path

# 1. หา Path ปัจจุบันที่รัน Script อยู่
current_path = Path.cwd()
# current_path.parent.parent/WebCPITH เซฟในนี้
# 2. กำหนดชื่อโฟลเดอร์ตามเงื่อนไข (ชุดทั่วไป หรือ ชุดนอกเขต)
folder_name = "G" if text == 'g' else "U"
export_dir = current_path / folder_name

# 3. สร้างโฟลเดอร์ถ้ายังไม่มี (exist_ok=True คือถ้ามีอยู่แล้วจะไม่ Error)
export_dir.mkdir(parents=True, exist_ok=True)

# 4. เตรียมข้อมูลสำหรับ JSON (Metadata + Data)
export_package = {
    "metadata": {
        "year_month": f"{Year}/{Month}",
        "display_date": f"ข้อมูลประจำเดือน {Month}/{Year}"
    },
    "data": save1.to_dict(orient='records')
}

# 5. บันทึกไฟล์ลงในโฟลเดอร์ที่สร้างไว้
file_path = export_dir / "my_data.json"

with open(file_path, 'w', encoding='utf-8') as f:
    json.dump(export_package, f, ensure_ascii=False, indent=2)

print(f"✅ บันทึกไฟล์ JSON เรียบร้อยที่: {file_path}")
print(f"📌 อย่าลืมย้ายไฟล์ index.html ไปไว้ในโฟลเดอร์ '{folder_name}' ด้วยเพื่อให้เรียกใช้งานได้")

✅ บันทึกไฟล์ JSON เรียบร้อยที่: d:\Coding\WorkFlow\04-Excess_Lost\G\my_data.json
📌 อย่าลืมย้ายไฟล์ index.html ไปไว้ในโฟลเดอร์ 'G' ด้วยเพื่อให้เรียกใช้งานได้


In [20]:
import json
import os
from pathlib import Path


# 1. จัดการ Path: ถอยกลับ 2 ระดับ แล้วเข้าโฟลเดอร์ WebCPITH
base_path = Path.cwd().parent.parent / "WorkWeb"

# 2. กำหนดชื่อโฟลเดอร์ลูกตามเงื่อนไข
folder_name = "G" if text == 'g' else "U"
export_dir = base_path / folder_name

# 3. สร้างโฟลเดอร์ (รวมถึง WebCPITH ถ้ายังไม่มี)
export_dir.mkdir(parents=True, exist_ok=True)

# 4. เตรียมข้อมูลสำหรับ JSON
export_package = {
    "metadata": {
        "year_month": f"{Year}/{Month}",
        "display_date": f"ข้อมูลประจำเดือน {Month}/{Year}"
    },
    "data": save1.to_dict(orient='records')  # แปลง DataFrame เป็น List of Dict
}

# 5. บันทึกไฟล์
file_path = export_dir / "my_data.json"

with open(file_path, 'w', encoding='utf-8') as f:
    json.dump(export_package, f, ensure_ascii=False, indent=2)

print(f"✅ บันทึกไฟล์เรียบร้อยที่: {file_path}")

✅ บันทึกไฟล์เรียบร้อยที่: d:\Coding\WorkWeb\G\my_data.json
